# LAB: Practice with Tavily + LangChain Agents

### Objective
Reinforce your understanding of LangChain agents integrated with Tavily for real-time web search by solving two open-ended tasks. You'll:
- Load and configure a LangChain agent with Tavily
- Build effective prompts
- Generate structured, useful outputs from real-time data

### Setup

In [5]:
import langchain
print(langchain.__version__)

0.3.30


In [1]:
import os
from langchain.agents import initialize_agent, AgentType, Tool
from langchain.chat_models import ChatOpenAI

In [61]:
import os
from langchain.agents import initialize_agent, AgentType, Tool
from langchain.chat_models import ChatOpenAI
from langchain.tools.tavily_search import TavilySearchResults
import tiktoken
from IPython.display import Markdown, display
from  dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')
from langchain.tools.tavily_search import TavilySearchResults

load_dotenv()
tavily_search = TavilySearchResults()

# LLM + Encoding
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
encoding = tiktoken.encoding_for_model("gpt-4o-mini")

# Safe Tavily wrapper
def safe_search(query: str) -> str:
    result = tavily_search.run(query)

    # Ensure result is a string — Tavily returns dict with 'snippets' sometimes
    if isinstance(result, dict):
        result_text = result.get("content", "") or str(result)
    else:
        result_text = str(result)

    tokens = encoding.encode(result_text)
    trimmed = encoding.decode(tokens[:3500])  # leave room for GPT-4 response
    return trimmed


# LangChain tool
tools = [Tool(name="TavilySafeSearch", func=safe_search, description="Web search tool")]
agent = initialize_agent(tools=tools, llm=llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True, handle_parsing_errors=True,max_execution_time = 60, early_stopping_method="generate",return_intermediate_steps=True)

### Exercise 1: AI in Healthcare

Goal: Investigate and summarize the latest advancements in generative AI applied to healthcare in 2025.

- Design a prompt that asks the agent to retrieve the most recent updates.
- Ensure the agent outputs a structured response in Markdown.

In [3]:
result = tavily_search.invoke("latest updates in generative AI applied to healthcare")

In [4]:
display(result)

[{'title': 'Generative AI to Reshape the Future of Health Care | Deloitte US',
  'url': 'https://www.deloitte.com/us/en/industries/life-sciences-health-care/articles/generative-ai-in-healthcare.html',
  'content': 'In recent years, natural language processing and machine learning have found applications in various health care use cases, but new Generative AI models are taking health care technology to new heights. These models demonstrate unprecedented capabilities in natural language generation, summarization, translation, insight retrieval, reasoning, and managing unstructured, unlabeled data. Generative AI technology has the potential to democratize knowledge, increase interoperability, accelerate discovery, and enable true personalization in health care.\n\n### Consumer trust hangs in the balance [...] Generative AI in health care also holds promise across industries to streamline operations, from discovery through commercialization—enhancing efficiency, compliance, and consumer-ce

In [22]:
# Your prompt here
prompt_1 = """ 

1. Model's role: 
You must retrieve the latests advancements of generative AI applied in the field of healthcare.(score > 0.7). (URL and content)

2. Tone: Use a unbiased and journalistic tone. 

3. Task description: 
    - Cite one quote from the 3 most relevant items retrieved where the authors explain the impact of the method. 
    
4. Formatting. 
    - Write the content with markdown style. 
    - Use bulletpoints to separate the relevant retrieved elements to improve readability. 
    - At the end of each element include the URL as reference. 

"""

# Run it
response_1 = agent.run(prompt_1)




> Entering new AgentExecutor chain...
I need to search for the latest advancements in generative AI applied in healthcare to gather relevant information and quotes. 

Action: TavilySafeSearch  
Action Input: "latest advancements in generative AI in healthcare"  

Observation: [{'title': '5 Gen AI Trends for Healthcare Heading into 2026', 'url': 'https://purelogics.com/5-gen-ai-trends-for-healthcare/', 'content': 'Table of Contents\n\nGenerative AI in healthcare is set to move deeper into care delivery and operations, and market momentum reflects this shift. The Precedence Research study, based on industry publications, company reports, and analysis of market drivers such as investment trends and AI adoption, indicates that the gen AI market in healthcare will reach $40 billion by 2034.\n\nGenerative AI in healthcare Market size 2023 to 2034\n\nBesides the McKinsey & Company survey of US healthcare leaders found that generative AI is perceived to deliver the greatest value in three ma

In [20]:
display(Markdown(response_1))


### Relevant Advancements in Generative AI in Healthcare

- **Article: Generative AI to Reshape the Future of Health Care | Deloitte US**  
  **Quote:** "Generative AI offers revolutionary approaches that can help secure a competitive advantage in the health care sector—and re-establish genuine care and trust in health care practices."  
  **URL:** [Deloitte](https://www.deloitte.com/us/en/industries/life-sciences-health-care/articles/generative-ai-in-healthcare.html)

- **Article: Recent Advances in AI and GenAI for Health Informatics**  
  **Quote:** "AI-assisted RPM systems have significantly advanced the delivery of care by enabling continuous, tailored data monitoring and processing in real time."  
  **URL:** [PMC](https://pmc.ncbi.nlm.nih.gov/articles/PMC12941142/)

- **Article: An AI model beat doctors at diagnosing patients, in a new study : NPR**  
  **Quote:** "You have something which is quite accurate, possibly ready for prime time," he says. "Now the open question is how the heck do you introduce it into clinical workflows in ways that actually improve care?"  
  **URL:** [NPR](https://www.npr.org/2026/04/30/nx-s1-5804474/ai-doctors-openai-patient-care-diagnosis)

### Conclusion
These articles highlight the transformative potential of generative AI in healthcare, emphasizing its ability to enhance research, personalize medicine, and improve operational efficiency.

### Exercise 2: AI Startups Landscape

Goal: Track 2025’s top emerging AI startups and their innovations.

- Create a prompt that instructs the agent to deliver a clean Markdown summary.
- Tip: Ask for company names, product highlights, and sources.

In [ ]:
# Your prompt here
input_2 = "Retrieve the top 2025 emerging AI startups and their innovations"
prompt_2 = """

Answer the question as best as you can. You have access to the following tools: 
{tools}

Follow this format: 

Question: {input_2}
Thought: Think what to do. Do I need to use a tool?
Action: take an action from [{tool_names}]
Action input: the input to the action
Observation: the result of the action

Repeat the cycle Thought/Action/Action input/Observation N times

When you gathered enough information to answer, use the following format: 
    
- Answer with an unbiased and journalistic tone. 
- Write your answers with markdown style. 
- Use bulletpoints to separate the relevant retrieved elements to improve readability. 
- For each element include: 
    1. Name of the Startup 
    2. Type of application and impact of the innovation in less than 200 words.
    3. URL as reference. 

{agent_scratchpad}
"""

# Run it
response_2 = agent.run(prompt_2)




> Entering new AgentExecutor chain...
Question: What are some innovative startups in the field of health technology that have emerged recently?

Thought: I need to gather information about recent health technology startups and their innovations. Using a web search tool will help me find relevant and up-to-date information.

Action: TavilySafeSearch
Action Input: "recent health technology startups 2023"

Observation: [{'title': '2023 North America Health Tech 200', 'url': 'https://www.holoniq.com/notes/2023-north-america-health-tech-200', 'content': 'High profile failures and a general market slowdown across digital health has not prevented a strong crop of startups from emerging in HolonIQ’s 2023 North America Health Tech 200. The region’s startup scene continues to attract entrepreneurs seeking to address North America’s substantial healthcare challenges and, despite headwinds, the 2023 cohort includes some of the leading health startups globally. Rising expectations around the near

In [25]:
display(Markdown(response_2))


### Innovative Startups in Health Technology (2023)

- **Capital RX**
  - **Type of Application**: Pharmacy benefit manager that aims to reduce prescription drug costs and improve transparency in the pharmaceutical supply chain.
  - **Impact**: By leveraging technology to streamline pharmacy benefits, Capital RX enhances patient access to medications while lowering costs for employers and patients.
  - **URL**: [HolonIQ](https://www.holoniq.com/notes/2023-north-america-health-tech-200)

- **Headway**
  - **Type of Application**: A mental health platform that connects patients with therapists and mental health professionals.
  - **Impact**: Headway addresses the growing demand for mental health services by simplifying the process of finding and accessing care, thus improving mental health outcomes.
  - **URL**: [HolonIQ](https://www.holoniq.com/notes/2023-north-america-health-tech-200)

- **Evvy**
  - **Type of Application**: A point-of-care testing startup focused on women's health (FemTech).
  - **Impact**: Evvy provides accessible and accurate testing for women's health issues, empowering women to take charge of their health with timely information.
  - **URL**: [HolonIQ](https://www.holoniq.com/notes/2023-north-america-health-tech-200)

- **Aledade**
  - **Type of Application**: A primary care physician platform that supports independent practices in managing patient care.
  - **Impact**: Aledade enhances the quality of care by providing tools and resources for primary care physicians to improve patient outcomes and reduce costs.
  - **URL**: [HolonIQ](https://www.holoniq.com/notes/2023-north-america-health-tech-200)

- **Diligent Robotics**
  - **Type of Application**: Robotics in healthcare, specifically focused on creating robot assistants for hospitals.
  - **Impact**: Diligent Robotics aims to alleviate staff shortages and improve operational efficiency in healthcare settings by deploying robots to assist with routine tasks.
  - **URL**: [HolonIQ](https://www.holoniq.com/notes/2023-north-america-health-tech-200)

These startups represent a diverse range of innovations in health technology, addressing various challenges in the healthcare system while leveraging modern technology to improve patient care and outcomes.

### Exercice 3: Compare Two Tech Products
- **Prompt idea:** Compare the key features, pricing, and reviews of OpenAI’s ChatGPT Team and Anthropic’s Claude Pro.
- Ensure the agent outputs a structured response in Markdown.




In [62]:
# Your prompt here
input_3 = "Compare the key features, pricing and reviews of OpenAI's ChatGPT team and Anthropic's Claude Pro."
prompt_3 = f"""

Answer the question as best as you can. You have access to the following tools: 
{tools}

Follow this format: 

Question: {input_3}
Thought: Think what to do. Do I need to use a tool?
Action: take an action from [{tools}]
Action input: the input to the action
Observation: the result of the action

Repeat the cycle Thought/Action/Action input/Observation N times

When you gathered enough information to answer, show your response in the following format: 
    
- Write the final result in a table format using markdown style: 
rows=compared features 
columns=[OpenAI's ChatGPT,Anthropic's Claude Pro] 
Finished chain contains the full context and write the answer.

"""

# Run it
response_3 = agent.invoke(prompt_3)




> Entering new AgentExecutor chain...
Question: Compare the key features, pricing and reviews of OpenAI's ChatGPT team and Anthropic's Claude Pro.
Thought: I need to gather information about the key features, pricing, and reviews for both OpenAI's ChatGPT and Anthropic's Claude Pro. This will require a web search to find the most recent and relevant data.
Action: TavilySafeSearch
Action Input: "OpenAI ChatGPT features pricing reviews"
Observation: [{'title': 'ChatGPT Plans Compared: Free vs Plus ($20) vs Pro ($200) vs ...', 'url': 'https://intuitionlabs.ai/articles/chatgpt-plans-comparison', 'content': 'Future Capabilities: Upcoming features will also differentiate plans. For example, as ChatGPT gains real-time data connections, new LLM improvements, or integrated multimodal sensors, OpenAI will likely gate advanced features to higher-tier subscribers at first. This means the gap between editions may widen over time. Enterprise customers will push for continued expansion of context (

In [63]:
response_3.keys()


dict_keys(['input', 'output', 'intermediate_steps'])

In [73]:
display(Markdown(response_3["output"]))


### Key Features and Pricing Comparison

| Feature/Aspect                | OpenAI's ChatGPT                                      | Anthropic's Claude Pro                               |
|-------------------------------|------------------------------------------------------|-----------------------------------------------------|
| **Key Features**              | - Access to GPT-4 and GPT-5 models                   | - Advanced reasoning and coding capabilities         |
|                               | - Collaboration tools for teams                       | - Computer use and browser automation                |
|                               | - Custom GPTs and admin tools for businesses         | - Claude Code for coding and file execution         |
|                               | - Priority access to new features                     | - Integration with Google Workspace                  |
|                               | - Advanced voice mode and faster response times      | - Enhanced reasoning models                          |
| **Pricing**                   | - Free: Limited access                               | - Free: Limited access                               |
|                               | - Plus: $20/month                                    | - Pro: $20/month (annual) or $25/month (monthly)   |
|                               | - Business: $25-$30/user/month                       | - Max 5x: $100/month                                 |
|                               | - Enterprise: Custom pricing                         | - Max 20x: $200/month                                |
|                               |                                                      | - Team Standard: $30/user/month                      |
|                               |                                                      | - Team Premium: $150/user/month                      |
| **Reviews**                   | - Generally positive, noted for versatility          | - Positive feedback on formal writing capabilities    |
|                               | - Users appreciate the advanced features in paid tiers| - Users highlight cost efficiency and performance    |
|                               | - Some concerns about pricing increases in the future | - Praised for its dual-revenue model and flexibility |

### Summary
- **OpenAI's ChatGPT** offers a range of features suitable for both individual users and businesses, with a tiered pricing model that includes a free version and several paid options. Users appreciate its versatility and advanced capabilities, although there are concerns about potential future price increases.
  
- **Anthropic's Claude Pro** is positioned as a strong competitor, particularly for users needing advanced reasoning and coding capabilities. Its pricing is competitive, and it offers unique features like browser automation and integration with Google Workspace. Reviews highlight its efficiency and effectiveness in formal writing tasks.

This comparison provides a clear overview of the two platforms, helping users make informed decisions based on their specific needs and budget.

## Bonus Task: Propose and Implement Your Own Use Case

As a final challenge, think of a real-world scenario where an AI agent could provide value using web search or external tools.
